In [1]:
from steer_core.DataManager import DataManager

from steer_materials.CellMaterials.Base import CurrentCollectorMaterial, InsulationMaterial, SeparatorMaterial
from steer_materials.CellMaterials.Electrode import CathodeMaterial, Binder, ConductiveAdditive, AnodeMaterial

from steer_opencell_design.Components.CurrentCollectors import TablessCurrentCollector
from steer_opencell_design.Formulations.ElectrodeFormulations import CathodeFormulation, AnodeFormulation
from steer_opencell_design.Components.Electrodes import Cathode, Anode
from steer_opencell_design.Components.Separators import Separator
from steer_opencell_design.Constructions.Layups import Laminate
from steer_opencell_design.Constructions.ElectrodeAssemblies.JellyRolls import WoundJellyRoll
from steer_opencell_design.AuxillaryComponents.WindingEquipment import RoundMandrel

from steer_opencell_design import __version__

import pandas as pd
from datetime import datetime as dt

In [2]:
db = DataManager()

In [3]:
db.remove_data(table_name='cells', condition="name = 'UniGrid-NCO32140'")

In [4]:
db.get_table_names()

['cells',
 'anode_materials',
 'binder_materials',
 'cathode_materials',
 'conductive_additive_materials',
 'current_collector_materials',
 'insulation_materials',
 'separator_materials']

In [5]:
# Get current collector materials from the database

current_collector_material = CurrentCollectorMaterial.from_database('Aluminum')
conductive_additive = ConductiveAdditive.from_database("Super P")
binder = Binder.from_database("PVDF")
insulation_material = InsulationMaterial.from_database("Aluminium Oxide, 95%")
separator_material = SeparatorMaterial.from_database("Polypropylene")


In [6]:
# Create the cathode

cathode_current_collector = TablessCurrentCollector(
    material=current_collector_material,
    width=130,
    length=2635,
    coated_width=124,
    insulation_width=2.5,
    thickness=12
)


cathode_active_material = CathodeMaterial.from_database("NaNiMn P2-O3 Composite")

cathode_formulation = CathodeFormulation(
    active_materials={cathode_active_material: 95},
    binders={binder: 2.5},
    conductive_additives={conductive_additive: 2.5}
)

my_cathode = Cathode(
    formulation=cathode_formulation,
    current_collector=cathode_current_collector,
    calender_density=2.64,
    mass_loading=18.18,
    insulation_material=insulation_material,
    insulation_thickness=3
)


In [7]:
# Create the anode

anode_current_collector = TablessCurrentCollector(
    material=current_collector_material,
    width=134,
    length=2640,
    coated_width=128,
    insulation_width=2.5,
    thickness=12
)

anode_active_material = AnodeMaterial.from_database("Hard Carbon (Vendor B)")

anode_formulation = AnodeFormulation(
    active_materials={anode_active_material: 95},
    binders={binder: 2.5},
    conductive_additives={conductive_additive: 2.5}
)

my_anode = Anode(
    formulation=anode_formulation,
    current_collector=anode_current_collector,
    calender_density=1.05,
    mass_loading=9,
    insulation_material=insulation_material,
    insulation_thickness=3
)


In [8]:
# make the layup

top_separator = Separator(
    material=separator_material,
    width=135,
    length=2800,
    thickness=25
)

bottom_separator = Separator(
    material=separator_material,
    width=135,
    length=2800,
    thickness=25
)

my_layup = Laminate(
    anode=my_anode,
    cathode=my_cathode,
    top_separator=top_separator,
    bottom_separator=bottom_separator,
    name="UniGrid-NCO32140",
)


In [9]:
# create the jellyroll assembly

mandrel = RoundMandrel(
    diameter=5,
    length=500,
)

my_jellyroll = WoundJellyRoll(
    laminate=my_layup,
    mandrel=mandrel,
    name="UniGrid-NCO32140",
)


In [10]:
my_jellyroll.roll_properties

,Turns
Component,
anode_a_side_coating_turns,40.18
anode_current_collector_turns,40.18
anode_b_side_coating_turns,40.18
cathode_a_side_coating_turns,39.97
cathode_current_collector_turns,39.97
cathode_b_side_coating_turns,39.97
bottom_separator_turns,45.80
bottom_separator_inner_turns,4.90
bottom_separator_outer_turns,0.70


In [11]:
my_jellyroll.get_spiral_plot(height=1300, width=1300).show()

In [12]:

db.insert_data(table_name='cells', data=pd.DataFrame({
    'name': [my_jellyroll.name],
    'object': [my_jellyroll.serialize()],
    'form_factor': ['Cylindrical'],
    'internal_construction': ['Wound'],
    'date': [dt.now().strftime("%Y-%m-%d %H:%M:%S")],
    'version': [__version__],
    'reference': ['Na/Na+']
}))


In [13]:
db.get_data('cells')

,name,object,form_factor,internal_construction,date,version,reference
0,Vendor D NFPP,gASVF/0AAAAAAACMKnN0ZWVyX29wZW5jZWxsX2Rlc2lnbi...,Prismatic,Stacked,2025-10-21 15:18:45,0.3.12,Na/Na+
1,Vendor E NFPP,gASVF/0AAAAAAACMKnN0ZWVyX29wZW5jZWxsX2Rlc2lnbi...,Prismatic,Stacked,2025-10-21 15:18:46,0.3.12,Na/Na+
2,Vendor G NFM,gASVswMAAAAAAACMKnN0ZWVyX29wZW5jZWxsX2Rlc2lnbi...,Cylindrical,Wound,2025-10-21 15:18:48,0.3.12,Na/Na+
3,Vendor G NFPP,gASVaPQAAAAAAACMKnN0ZWVyX29wZW5jZWxsX2Rlc2lnbi...,Cylindrical,Wound,2025-10-21 15:18:50,0.3.12,Na/Na+
4,CBAK-32140NS,gASVOAwAAAAAAACMQnN0ZWVyX29wZW5jZWxsX2Rlc2lnbi...,Cylindrical,Wound,2025-11-03 16:48:50,0.3.12,Na/Na+
5,Cell 2,gASVTAEBAAAAAACMQnN0ZWVyX29wZW5jZWxsX2Rlc2lnbi...,Cylindrical,Wound,2025-11-03 16:49:50,0.3.12,Na/Na+
6,HiNa-NaCR32140-MP10,gASVOAwAAAAAAACMQnN0ZWVyX29wZW5jZWxsX2Rlc2lnbi...,Cylindrical,Wound,2025-11-04 08:14:15,0.3.12,Na/Na+
7,QNAS-S40160NL,gASVOAwAAAAAAACMQnN0ZWVyX29wZW5jZWxsX2Rlc2lnbi...,Cylindrical,Wound,2025-11-04 08:16:17,0.3.12,Na/Na+
8,QNAS-S40160RL,gASVOAwAAAAAAACMQnN0ZWVyX29wZW5jZWxsX2Rlc2lnbi...,Cylindrical,Wound,2025-11-04 08:18:00,0.3.12,Na/Na+
9,UniGrid-NCO32140,gASVMDIBAAAAAACMQnN0ZWVyX29wZW5jZWxsX2Rlc2lnbi...,Cylindrical,Wound,2025-11-04 08:19:09,0.3.12,Na/Na+
